In [ ]:
import os
import pandas as pd
import numpy as np

class IntegratedHealthcareHTAPipeline:
    def __init__(self):
        # 1. Establish strict Nova Scotia Health Zone Boundaries
        self.zones = ["Central Zone", "Eastern Zone", "Northern Zone", "Western Zone"]
        self.conditions = ["Diabetes Mellitus Type II", "COPD Chronic Care", "Congestive Heart Failure (CHF)"]

        # 2. Strict 10-Subcategory Comprehensive MCDA Weight Matrix (Sums to 1.0)
        self.mcda_weights = {
            "W_clinical_efficacy": 0.10, "W_frontline_workflow": 0.10, "W_waittime_mitigation": 0.10,
            "W_cost_utility_icer": 0.10, "W_total_cost_ownership": 0.10, "W_downstream_offset": 0.10,
            "W_geographic_parity": 0.10, "W_cultural_inclusivity": 0.10, "W_infrastructure_interop": 0.05,
            "W_environmental_footprint": 0.075, "W_contractual_resilience": 0.075
        }

        # 3. NSNU Collective Agreement Salary Matrix Configurations
        self.nsnu_hourly_rates = {"LPN": 35.81, "RN": 49.99, "NP": 69.52}
        self.annual_hours = 1950

        # Localized Zone Multipliers (Regional Premium Factors)
        self.zone_multipliers = {
            "Central Zone": 1.04,   # Tertiary specialization shift
            "Eastern Zone": 0.99,
            "Northern Zone": 0.98,
            "Western Zone": 1.00
        }

        # 4. Infrastructure & Acute Care Financial Constants
        self.base_laptop_cost = 1425.00
        self.cost_per_ed_visit = 380.00
        self.cost_per_admission = 6800.00  # Based on 5.2-day average Length of Stay (LOS)

        # Baseline annual acute utilization assumptions per 100 chronic patients before intervention
        self.baseline_annual_ed_visits_per_100 = 85
        self.baseline_annual_admissions_per_100 = 25

        # Preventative intervention efficacy rates
        self.ed_reduction_rate = 0.15
        self.admission_reduction_rate = 0.20

    def calculate_annual_team_salary(self, lpn_count, rn_count, np_count, zone_multiplier, contract_inflation_factor):
        """Calculates a combined annualized salary incorporating regional multipliers and contract steps."""
        lpn_total = lpn_count * (self.nsnu_hourly_rates["LPN"] * self.annual_hours)
        rn_total  = rn_count  * (self.nsnu_hourly_rates["RN"]  * self.annual_hours)
        np_total  = np_count  * (self.nsnu_hourly_rates["NP"]  * self.annual_hours)
        base_team_salary = (lpn_total + rn_total + np_total) * contract_inflation_factor
        return round(base_team_salary * zone_multiplier, 2)

    def calculate_split_system_savings(self, target_population):
        """Computes acute care grid savings explicitly broken down by triage and admission."""
        # Expected baseline events without remote care intervention
        expected_ed_baseline = (target_population * self.baseline_annual_ed_visits_per_100) / 100
        expected_adm_baseline = (target_population * self.baseline_annual_admissions_per_100) / 100

        # Absolute events averted via preventative monitoring
        ed_visits_prevented = expected_ed_baseline * self.ed_reduction_rate
        admissions_prevented = expected_adm_baseline * self.admission_reduction_rate

        # Translate averted events into distinct financial metrics
        ed_triage_savings = round(ed_visits_prevented * self.cost_per_ed_visit, 2)
        inpatient_admission_savings = round(admissions_prevented * self.cost_per_admission, 2)
        total_gross_savings = round(ed_triage_savings + inpatient_admission_savings, 2)

        return ed_triage_savings, inpatient_admission_savings, total_gross_savings

    def execute_and_export_all(self):
        """Executes the end-to-end data engineering pipeline for both datasets."""
        np.random.seed(42)
        os.makedirs("generated", exist_ok=True)

        # --- PART A: PROCESS CORE MCDA DATA LAYER ---
        mcda_rows = []
        lpn_comp, rn_comp, np_comp = self.nsnu_hourly_rates["LPN"] * 0.30, self.nsnu_hourly_rates["RN"] * 0.60, self.nsnu_hourly_rates["NP"] * 0.10
        blended_base_rate = lpn_comp + rn_comp + np_comp

        for zone in self.zones:
            multiplier = self.zone_multipliers[zone]
            derived_rate = round(blended_base_rate * multiplier, 2)
            cohort_base = 4200 if zone == "Central Zone" else (2100 if zone == "Eastern Zone" else (1850 if zone == "Northern Zone" else 2400))

            for condition in self.conditions:
                patients = int(cohort_base * np.random.uniform(0.25, 0.40))
                laptops = int(np.ceil(patients * 0.04)) + 1
                nurse_hours = int(patients * np.random.uniform(10.0, 16.0))
                total_tco = (nurse_hours * derived_rate) + (laptops * self.base_laptop_cost)

                mcda_rows.append({
                    "Zone": zone, "TargetPopulation": condition, "PatientCohortSize": patients,
                    "ClinicalEfficacy_Safety": round(np.random.uniform(70.0, 95.0), 1),
                    "FrontlineWorkflowImpact": round(np.random.uniform(65.0, 90.0), 1),
                    "WaitTimeMitigation": round(np.random.uniform(15.0, 45.0), 1),
                    "CostUtility_ICER": round(total_tco / patients, 2), "TotalCostOwnership": round(total_tco, 2),
                    "DownstreamCostOffset": round(np.random.uniform(25000.0, 85000.0), 2), "GeographicParity": round(np.random.uniform(1.5, 4.9), 2),
                    "CulturalStructuralInclusivity": round(np.random.uniform(70.0, 98.0), 1), "InfrastructureInteroperability": round(np.random.uniform(60.0, 95.0), 1),
                    "EnvironmentalFootprint": round(np.random.uniform(10.0, 40.0), 1), "ContractualOperationalResilience": round(np.random.uniform(75.0, 95.0), 1)
                })

        df_mcda = pd.DataFrame(mcda_rows)
        zone_summary = df_mcda.groupby("Zone").agg({
            "PatientCohortSize": "sum", "ClinicalEfficacy_Safety": "mean", "FrontlineWorkflowImpact": "mean", "WaitTimeMitigation": "mean",
            "CostUtility_ICER": "mean", "TotalCostOwnership": "sum", "DownstreamCostOffset": "mean", "GeographicParity": "mean",
            "CulturalStructuralInclusivity": "mean", "InfrastructureInteroperability": "mean", "EnvironmentalFootprint": "mean", "ContractualOperationalResilience": "mean"
        }).reset_index()

        def norm_b(col): return (col - col.min()) / (col.max() - col.min()) if (col.max() - col.min()) != 0 else 1.0
        def norm_c(col): return (col.max() - col) / (col.max() - col.min()) if (col.max() - col.min()) != 0 else 1.0

        zone_summary["N_Efficacy"] = norm_b(zone_summary["ClinicalEfficacy_Safety"])
        zone_summary["N_Workflow"] = norm_b(zone_summary["FrontlineWorkflowImpact"])
        zone_summary["N_WaitTime"] = norm_b(zone_summary["WaitTimeMitigation"])
        zone_summary["N_ICER"] = norm_c(zone_summary["CostUtility_ICER"])
        zone_summary["N_TCO"] = norm_c(zone_summary["TotalCostOwnership"])
        zone_summary["N_Offset"] = norm_b(zone_summary["DownstreamCostOffset"])
        zone_summary["N_GeoParity"] = norm_b(zone_summary["GeographicParity"])
        zone_summary["N_Inclusivity"] = norm_b(zone_summary["CulturalStructuralInclusivity"])
        zone_summary["N_Interop"] = norm_b(zone_summary["InfrastructureInteroperability"])
        zone_summary["N_Carbon"] = norm_b(zone_summary["EnvironmentalFootprint"])
        zone_summary["N_Resilience"] = norm_b(zone_summary["ContractualOperationalResilience"])

        zone_summary["Pillar_1_Clinical_Quality_Score"] = ((zone_summary["N_Efficacy"] * self.mcda_weights["W_clinical_efficacy"]) + (zone_summary["N_Workflow"] * self.mcda_weights["W_frontline_workflow"]) + (zone_summary["N_WaitTime"] * self.mcda_weights["W_waittime_mitigation"])) * 100
        zone_summary["Pillar_2_Fiscal_Accountability_Score"] = ((zone_summary["N_ICER"] * self.mcda_weights["W_cost_utility_icer"]) + (zone_summary["N_TCO"] * self.mcda_weights["W_total_cost_ownership"]) + (zone_summary["N_Offset"] * self.mcda_weights["W_downstream_offset"])) * 100
        zone_summary["Pillar_3_Health_Equity_Score"] = ((zone_summary["N_GeoParity"] * self.mcda_weights["W_geographic_parity"]) + (zone_summary["N_Inclusivity"] * self.mcda_weights["W_cultural_inclusivity"]) + (zone_summary["N_Interop"] * self.mcda_weights["W_infrastructure_interop"])) * 100
        zone_summary["Pillar_4_System_Sustainability_Score"] = ((zone_summary["N_Carbon"] * self.mcda_weights["W_environmental_footprint"]) + (zone_summary["N_Resilience"] * self.mcda_weights["W_contractual_resilience"])) * 100

        zone_summary["Composite_MCDA_Index"] = zone_summary["Pillar_1_Clinical_Quality_Score"] + zone_summary["Pillar_2_Fiscal_Accountability_Score"] + zone_summary["Pillar_3_Health_Equity_Score"] + zone_summary["Pillar_4_System_Sustainability_Score"]
        zone_summary = zone_summary.sort_values(by="Composite_MCDA_Index", ascending=False).reset_index(drop=True)
        zone_summary["StrategicRank"] = zone_summary.index + 1

        for subcat, val in self.mcda_weights.items(): zone_summary[f"Config_Weight_{subcat}"] = val
        zone_summary.round(4).to_csv("generated/ns_health_mcda_pillars_output.csv", index=False)
        print("Success: Generated 'generated/ns_health_mcda_pillars_output.csv'")

        # --- PART B: COMPILER FOR 10-YEAR PHASED FORECAST WITH DETAILED SAVINGS ---
        financial_timeline = []
        hub_lpn, hub_rn, hub_np = 3, 2, 1
        total_hub_staff = hub_lpn + hub_rn + hub_np

        for year in range(1, 11):
            operational_inflation = (1 + 0.025) ** (year - 1)
            contract_cycles = (year - 1) // 4
            contract_inflation_factor = (1 + 0.03) ** contract_cycles

            # --- PHASE 1: CENTRALIZED CORE HUB OPERATIONS (YEARS 1 - 3) ---
            if year <= 3:
                global_reachable_cohort = 450
                cohort_per_zone = global_reachable_cohort / len(self.zones)

                hub_labor = self.calculate_annual_team_salary(hub_lpn, hub_rn, hub_np, 1.0, contract_inflation_factor) * operational_inflation
                split_labor_per_zone = round(hub_labor / len(self.zones), 2)

                global_laptop_capital = (total_hub_staff * self.base_laptop_cost) if year == 1 else 0.0
                split_capital_per_zone = round(global_laptop_capital / len(self.zones), 2)

                # Dynamic Savings Breakdown Calling Closure
                ed_save, adm_save, total_save = self.calculate_split_system_savings(cohort_per_zone)

                for zone in self.zones:
                    total_zone_cost = split_labor_per_zone + split_capital_per_zone
                    financial_timeline.append({
                        "FiscalYear": f"Year {year}",
                        "DeploymentPhase": "Centralized Hub Pilot",
                        "Zone": zone,
                        "Monitored_Patient_Cohort": int(cohort_per_zone),
                        "Operational_Labour_Cost": split_labor_per_zone,
                        "Laptop_Capital_Cost": split_capital_per_zone,
                        "Total_Program_Cost": total_zone_cost,
                        "ED_Triage_Savings": ed_save,
                        "Inpatient_Admission_Savings": adm_save,
                        "Total_Expected_Savings": total_save,
                        "Net_Fiscal_Impact": round(total_save - total_zone_cost, 2)
                    })

            # --- PHASE 2: DECENTRALIZED REGIONAL SCALE-OUT (YEARS 4 - 10) ---
            else:
                cohort_growth_modifier = (1 + 0.015) ** (year - 4)
                cohort_per_zone = int(600 * cohort_growth_modifier)

                # Dynamic Savings Breakdown calling closure for regional operations
                ed_save, adm_save, total_save = self.calculate_split_system_savings(cohort_per_zone)

                for zone in self.zones:
                    multiplier = self.zone_multipliers[zone]
                    zone_labor_cost = self.calculate_annual_team_salary(hub_lpn, hub_rn, hub_np, multiplier, contract_inflation_factor) * operational_inflation

                    # Cycle IT laptops: Initial decentralized purchase in Year 4, cyclic refresh in Year 7
                    if year == 4 or year == 7:
                        zone_laptop_capital = (hub_lpn + hub_rn + hub_np) * self.base_laptop_cost
                    else:
                        zone_laptop_capital = 0.0

                    total_zone_cost = zone_labor_cost + zone_laptop_capital

                    financial_timeline.append({
                        "FiscalYear": f"Year {year}",
                        "DeploymentPhase": "Decentralized Regional Operations",
                        "Zone": zone,
                        "Monitored_Patient_Cohort": cohort_per_zone,
                        "Operational_Labour_Cost": round(zone_labor_cost, 2),
                        "Laptop_Capital_Cost": round(zone_laptop_capital, 2),
                        "Total_Program_Cost": round(total_zone_cost, 2),
                        "ED_Triage_Savings": ed_save,
                        "Inpatient_Admission_Savings": adm_save,
                        "Total_Expected_Savings": total_save,
                        "Net_Fiscal_Impact": round(total_save - total_zone_cost, 2)
                    })

        df_financial = pd.DataFrame(financial_timeline)
        df_financial.to_csv("generated/ns_health_10yr_financial_forecast.csv", index=False)
        print("Success: Generated 'generated/ns_health_10yr_financial_forecast.csv'")
        print("\n=== SYSTEM EXECUTION COMPLETED SUCCESSFULLY ===")

if __name__ == "__main__":
    pipeline = IntegratedHealthcareHTAPipeline()
    pipeline.execute_and_export_all()

#Thanks for checking out my code ^_^


Success: Generated 'generated/ns_health_mcda_pillars_output.csv'
Success: Generated 'generated/ns_health_10yr_financial_forecast.csv'

=== SYSTEM EXECUTION COMPLETED SUCCESSFULLY ===
